# 01: Label with SAM, train YOLO, check it for the robot

Tasks 1-4 of the task description. The setup cells below are given: read them, then run them.
**The experiments are yours** (a coding agent may help).

Every experiment has the same three steps:
1. ✍️ write down your prediction,
2. run it,
3. ✍️ explain the result, especially where your prediction was wrong.

In [ ]:
# Setup: installs what is missing (Colab / RunPod) and finds vision.py
import importlib.util, pathlib, subprocess, sys, time, urllib.request
for pkg, mod in [("transformers", "transformers"), ("ultralytics", "ultralytics"),
                 ("segmentation-models-pytorch", "segmentation_models_pytorch")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
if (pathlib.Path.cwd().parent / "vision.py").exists():   # opened from vision/notebooks or vision/sol
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
elif not pathlib.Path("vision.py").exists():            # Colab: fetch it from the course repo
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhaw-physical-ai/cas-pair/main/vision/vision.py", "vision.py")
from vision import *
device = get_device()
print("device:", device)

In [ ]:
images, instances, train, test = load_pennfudan()
test_images = [images[k] for k in test]
test_instances = [instances[k] for k in test]
sam = SamLabeler()      # SAM 2.1 base-plus, ~300 MB on first use

## Given helpers
From `vision.py`; read the docstrings with `help(...)`:
- `SamLabeler`: `sam(image, boxes=...)` or `sam(image, points=..., point_labels=...)` returns masks and SAM's own score
- `boxes_from_instances`, `instances_to_masks`: boxes and per-person masks from the human labels
- `iou`, `mean_iou`, `person_recall`, `seconds_per_image`: metrics
- `write_yolo_dataset`, `train_yolo`, `yolo_predict`: YOLO plumbing
- `UNetSegmenter`: a semantic U-Net (task 3)
- `show(ax, image, mask, boxes=..., points=..., point_labels=...)`: plotting

One SAM call, as an example:

In [ ]:
i = train[0]
boxes = boxes_from_instances(instances[i])
masks, scores = sam(images[i], boxes=boxes)
print(masks.shape, "| SAM's own guess:", scores.round(2), "| IoU vs human:", round(iou(masks.any(0), instances[i] > 0), 3))
fig, ax = plt.subplots(figsize=(6, 6))
show(ax, images[i], masks, boxes=boxes, title="SAM 2.1 from boxes")

## Task 1: SAM as a labeling tool
Pick 3-5 people from different images (`instances[i] == pid` is one person's human mask). Prompt
SAM with one click, then with a box, and compare each mask with the human mask.

**✍️ Your prediction (before running anything):** Which prompt gives the best mask: one click, a few clicks, or a box? Roughly what IoU?

> 

In [ ]:
# your code

Find a click that makes SAM segment the *wrong* thing (a bag, a shirt, two people), then fix
it with more clicks: positive (label 1) on what is missing, negative (label 0) on what is too much.

In [ ]:
# your code

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? Which prompt would you use to label 10 000 images?

> 

## Task 2: Label the training set
Turn the human **boxes** of all 120 training images into SAM masks: these are your labels. Keep
one `(n, H, W)` array per image, because YOLO needs one mask per person.

**✍️ Your prediction (before running anything):** Mean IoU of your labels against the human masks? How many images below 0.8?

> 

In [ ]:
# your code

Look at the 5 worst images. Who is wrong, SAM or the human?

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? Are SAM's labels good enough to train on?

> 

### A smaller SAM
SAM 2.1 comes in four sizes: tiny, small, base-plus (what you used so far), and large. Tiny has
31M parameters, base-plus 73M. Label the same 120 training images with
`SamLabeler("facebook/sam2.1-hiera-tiny")` and compare it with base-plus: mean IoU against the human masks, seconds per image, and the masks
side by side on the images where the two disagree most.

**✍️ Your prediction (before running anything):** How much label quality do you lose with tiny, and how much faster is it?

> 

In [ ]:
# your code

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? Which SAM would you use to label 100 000 images?

> 

## Task 3: Train your own YOLO
Write your SAM labels as a YOLO dataset and fine-tune `yolo11n-seg`, starting from weights
pretrained on COCO (~2-3 min on a T4):
```python
yaml = write_yolo_dataset("yolo_sam", [images[i] for i in train], sam_labels)
yolo_sam = train_yolo(yaml, name="sam", epochs=30)     # Ultralytics underneath; see help(train_yolo)
```
Then measure it on the 50 test images against the human masks:
`mean_iou(lambda im: yolo_predict(yolo_sam, im)[0].any(0), test_images, test_instances)`.

**✍️ Your prediction (before running anything):** Test IoU of YOLO trained on your SAM labels? (Your labels scored against the human masks in task 2.)

> 

In [ ]:
# your code

**✍️ Your prediction (before running anything):** The same YOLO trained on the *human* masks: how much better?

> 

In [ ]:
# your code

Look at test images with several people. What can a robot do with one mask per person that
"these pixels are person" cannot give it?

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? Was labeling with SAM a good idea?

> 

### A second kind of model: a semantic U-Net

**Two ways to segment.** YOLO does *instance* segmentation: it finds each person and gives each
one its own mask, box and confidence. A **semantic** segmenter answers a simpler question for
every pixel: person or not person? It returns one mask for the whole image and does not know
where one person ends and the next one begins.

**The U-Net.** The classic semantic segmenter, from medical imaging (2015). It has two halves:
- the **encoder** shrinks the image step by step (256 → 128 → ... → 8 pixels) and learns *what*
  is in it. Ours is a ResNet-18 pretrained on ImageNet, so it already knows edges, textures and
  shapes;
- the **decoder** grows the result back to full size and decides, pixel by pixel, *where* the
  person is. **Skip connections** hand it the fine details from the encoder at each size, which
  is why the outlines come out sharp.

The output is one number per pixel, turned into person / not person at 0.5. Training compares it
with the label, pixel by pixel (binary cross-entropy).

**In this lab:** `UNetSegmenter` from `vision.py` does all of that (images resized to 256 × 256,
25 epochs, ~30 s on a T4):
```python
unet = UNetSegmenter().fit(train_images, train_masks)   # one (H, W) bool mask per image
pred = unet(image)                                      # (H, W) bool
```
Train it on the **same SAM labels** as YOLO, merged into one mask per image (`m.any(0)`), and
measure it the same way. Then compare the three: SAM (needs a box per person), YOLO (instance)
and U-Net (semantic).

**✍️ Your prediction (before running anything):** Test IoU of the U-Net against YOLO? And which one would you put on the robot?

> 

In [ ]:
# your code

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? SAM, YOLO or U-Net: which is best, and for what?

> 

### Without a pretrained checkpoint

None of your models started from zero. `yolo11n-seg.pt` is a **checkpoint**: weights that
Ultralytics trained on COCO (118 000 images, 80 classes, *person* among them). The U-Net's encoder
starts from weights trained on ImageNet (1.2 million images). You only **fine-tuned** them on 120
images. Starting from weights learned on a big dataset is called **transfer learning**, and it is
the normal way to train a vision model.

What did the checkpoint give you? Train both models again on the same SAM labels, with the same
number of epochs, but from **random weights**:
- YOLO: the architecture file instead of the checkpoint,
  `train_yolo(yaml, name="scratch", model="yolo11n-seg.yaml", pretrained=False)`
- U-Net: `UNetSegmenter(encoder_weights=None)`

**✍️ Your prediction (before running anything):** Test IoU of YOLO and the U-Net trained from random weights?

> 

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? What would you need to train without a checkpoint?

> 

## Task 4: Is it ready for the robot?
A perception-planning-control loop runs at 10-30 Hz. And IoU hides what matters for safety: was
the person seen at all? `person_recall` counts the people of whom at least half the pixels were
found, and lists the ones it missed.

**✍️ Your prediction (before running anything):** Milliseconds per image for SAM and for YOLO? At which confidence threshold would you stop the robot?

> 

In [ ]:
# your code

In [ ]:
# your code

**✍️ Result and explanation:** What happened, and why? Which threshold, and what does it cost?

> 

## Report
Add one row per model to the class results table: labels (SAM / human), model, training images,
test IoU, `person_recall` at your chosen threshold, ms per image, GPU.